In [1]:
import os
import time
import tracemalloc
import re
from dataclasses import dataclass
from datetime import datetime, timedelta, time as dtime
from collections import defaultdict
from pathlib import Path

import pandas as pd
import numpy as np
import psutil
from gurobipy import Model, GRB, quicksum

# ============================================================
# 0. CONFIGURATION  — adjust paths as needed
# ============================================================

BASE_DIR = Path(r'C:\Users\emrec\Documents\parallel_scheduling\exactMEDIUM')

SHIPMENT_FILE       = BASE_DIR / 'sevkiyat_medium_selected8.xlsx'
MACHINE_GROUP_FILE  = BASE_DIR / 'machine_group_data_medium.xlsx'
SDST_FILE           = BASE_DIR / 'SDST.xlsx'

OUTPUT_SCHEDULE_FILE   = BASE_DIR / 'optimized_exact_medium_schedule.xlsx'
OUTPUT_VALIDATION_FILE = BASE_DIR / 'verification_exact_medium.xlsx'
GUROBI_LOG_FILE        = BASE_DIR / 'gurobi_exact_medium.log'   # <-- NEW: Gurobi log

print('Using files:')
print('SHIPMENT_FILE      =', SHIPMENT_FILE)
print('SDST_FILE          =', SDST_FILE)
print('MACHINE_GROUP_FILE =', MACHINE_GROUP_FILE)
print('GUROBI_LOG_FILE    =', GUROBI_LOG_FILE)

# Planning
PLANNING_START_HOUR = 7
DUE_TIME_HOUR       = 17
OVERTIME_START      = dtime(17, 0)
OVERTIME_END        = dtime(21, 0)
DISALLOW_START_IN_OVERTIME = True

# NOTE: Tardiness stays at 1.5 days — intentional project decision
TARDINESS_LIMIT_DAYS = 1.5
TARDINESS_LIMIT_MIN  = TARDINESS_LIMIT_DAYS * 24 * 60   # = 2160 min

ALLOW_JOB_SPLITTING = True
MAX_SPLITS    = 2
MIN_SPLIT_QTY = 100

INITIAL_SETUP_MIN           = 10.0
SAME_DIAM_SETUP_MIN         = 5.0
DEFAULT_DIFF_DIAM_SETUP_MIN = 20.0

W_OVERTIME  = 1.0
W_MAKESPAN  = 0.003
W_SETUP     = 0.01

MIP_GAP        = 0.05
TIME_LIMIT_SEC = 600
OUTPUT_FLAG    = 1

# REVISED Big-M:
# Schedule horizon = TARDINESS_LIMIT_MIN + 7*24*60 ≈ 10,080 + 2,160 = 12,240 min.
# The tightest valid upper bound for any start/finish time or
# no-overlap linearisation is 3x this horizon ≈ 36,720 min.
# We round up to 50,000 for a safe margin — still ~200x smaller than 10**7,
# which meaningfully tightens the LP relaxation and reduces solve time.
BIG_M = 50_000   # REVISED from 10**7

EPS = 1e-5


# ============================================================
# 1. HELPERS
# ============================================================

def normalize_machine_name(x):
    if pd.isna(x):
        return None
    s = str(x).strip().replace(' ', '').replace(',', '.').replace('O', '0')
    s = s.replace('T3.', 'T.3.')
    if re.match(r'^T3\d+', s):
        s = s.replace('T3', 'T.3.', 1)
    return s


def excel_serial_to_date(x):
    if pd.isna(x): return None
    if isinstance(x, datetime): return x.date()
    if isinstance(x, (int, float, np.integer, np.floating)):
        return (datetime(1899, 12, 30) + timedelta(days=float(x))).date()
    p = pd.to_datetime(x, errors='coerce')
    return None if pd.isna(p) else p.date()


def excel_time_to_td(x):
    if pd.isna(x): return None
    if isinstance(x, timedelta): return x
    if isinstance(x, datetime): return timedelta(hours=x.hour, minutes=x.minute, seconds=x.second)
    if isinstance(x, dtime): return timedelta(hours=x.hour, minutes=x.minute, seconds=x.second)
    if isinstance(x, (int, float, np.integer, np.floating)): return timedelta(days=float(x))
    p = pd.to_datetime(str(x), errors='coerce')
    return None if pd.isna(p) else timedelta(hours=p.hour, minutes=p.minute, seconds=p.second)


def combine_dt(date_val, time_val):
    d  = excel_serial_to_date(date_val)
    td = excel_time_to_td(time_val)
    if d is None or td is None: return None
    return datetime.combine(d, dtime(0, 0)) + td


def minutes_between(s, f):
    if f <= s: f += timedelta(days=1)
    return (f - s).total_seconds() / 60.0


def overlap_min(a_s, a_e, b_s, b_e):
    return max(0.0, (min(a_e, b_e) - max(a_s, b_s)).total_seconds() / 60.0)


def overtime_overlap(start_dt, end_dt):
    if end_dt <= start_dt: return 0.0
    total, day = 0.0, start_dt.date()
    while datetime.combine(day, dtime(0, 0)) <= end_dt:
        total += overlap_min(start_dt, end_dt,
                             datetime.combine(day, OVERTIME_START),
                             datetime.combine(day, OVERTIME_END))
        day += timedelta(days=1)
    return total


def parse_overtime_text(v):
    if pd.isna(v): return 0.0
    s = str(v).strip().lower()
    h = re.search(r'(\d+)\s*saat', s)
    m = re.search(r'(\d+)\s*dk', s)
    hours = int(h.group(1)) if h else 0
    mins  = int(m.group(1)) if m else 0
    if not h and not m:
        nums = re.findall(r'\d+', s)
        mins = int(nums[0]) if nums else 0
    return float(hours * 60 + mins)


def mode_or_first(series, default=None):
    s = series.dropna()
    if len(s) == 0: return default
    m = s.mode()
    return m.iloc[0] if len(m) > 0 else s.iloc[0]


def dt_to_min(dt, ps): return (dt - ps).total_seconds() / 60.0
def min_to_dt(m, ps):  return ps + timedelta(minutes=float(m))


# ============================================================
# 2. DATA CLASS
# ============================================================

@dataclass
class Job:
    job_id: str
    part_no: str
    due_date: datetime
    quantity: int
    diameter: float
    eligible_groups_op10: list
    eligible_groups_op20: list


# ============================================================
# 3. LOAD DATA
# ============================================================

def load_operations(path):
    raw = pd.read_excel(path, sheet_name=0, header=5, engine='openpyxl')
    raw.columns = [str(c).strip() for c in raw.columns]
    raw = raw.dropna(how='all')

    rename = {
        'Tarih': 'date', 'Başlangıç Saat': 'start_time', 'Bitiş Saat': 'finish_time',
        'Parça No': 'part_no', 'Makine No': 'machine',
        'CNC-1 operasyonu(piston)': 'op10_flag', 'CNC-2 operasyonu (saplama)': 'op20_flag',
        'Adet': 'quantity', 'Çap': 'diameter', 'Makine Grubu': 'machine_group',
        'Fazla mesai': 'overtime_text',
    }
    raw = raw.rename(columns=rename)
    raw = raw[raw['part_no'].notna()].copy()
    raw['machine']       = raw['machine'].apply(normalize_machine_name)
    raw['part_no']       = raw['part_no'].astype(int).astype(str)
    raw['quantity']      = pd.to_numeric(raw['quantity'], errors='coerce').fillna(0).astype(int)
    raw['diameter']      = pd.to_numeric(raw['diameter'], errors='coerce')
    raw['machine_group'] = pd.to_numeric(raw['machine_group'], errors='coerce').astype('Int64')
    raw['operation']     = np.where(raw.get('op10_flag', pd.Series(dtype=object)).notna(), 10,
                           np.where(raw.get('op20_flag', pd.Series(dtype=object)).notna(), 20, np.nan))
    raw = raw[raw['operation'].notna()].copy()
    raw['operation']     = raw['operation'].astype(int)
    raw['start_dt']      = [combine_dt(d, t) for d, t in zip(raw['date'], raw['start_time'])]
    raw['finish_dt']     = [combine_dt(d, t) for d, t in zip(raw['date'], raw['finish_time'])]
    raw['duration_min']  = [minutes_between(s, f) for s, f in zip(raw['start_dt'], raw['finish_dt'])]
    raw['unit_min']      = raw['duration_min'] / raw['quantity'].replace(0, np.nan)
    return raw


def load_orders(path):
    df = pd.read_excel(path, sheet_name=1, header=None, engine='openpyxl')
    jobs, current_date, in_block = [], None, False
    for _, row in df.iterrows():
        possible_date = None
        for val in row.tolist():
            if pd.isna(val): continue
            if isinstance(val, (int, float, np.integer, np.floating)) and 40000 < float(val) < 60000:
                possible_date = excel_serial_to_date(val); break
            if isinstance(val, datetime):
                possible_date = val.date(); break
        if possible_date is not None:
            current_date, in_block = possible_date, False; continue
        row_text = ' '.join([str(v).lower() for v in row.dropna().tolist()])
        if 'sipariş' in row_text and 'adet' in row_text:
            in_block = True; continue
        if in_block and current_date is not None:
            part      = row.iloc[4] if len(row) > 4 else None
            order_qty = row.iloc[5] if len(row) > 5 else None
            shipped   = row.iloc[6] if len(row) > 6 else None
            if pd.isna(part) or pd.isna(order_qty): continue
            try: part_no = str(int(part))
            except: continue
            qty = int(round(float(shipped if not pd.isna(shipped) else order_qty)))
            if qty <= 0: continue
            jobs.append({'job_id': f'{current_date.isoformat()}__{part_no}',
                         'part_no': part_no,
                         'due_date': datetime.combine(current_date, dtime(DUE_TIME_HOUR, 0)),
                         'quantity': qty})
    orders = pd.DataFrame(jobs)
    if orders.empty: raise ValueError('No orders found in Sayfa2.')
    before = len(orders)
    orders = orders.groupby(['job_id', 'part_no'], as_index=False).agg(
        quantity=('quantity', 'sum'), due_date=('due_date', 'min'))
    if len(orders) != before:
        print(f'Duplicates aggregated: {before} -> {len(orders)} unique jobs')
    return orders


def load_machine_groups(path):
    mg = pd.read_excel(path, sheet_name=0, engine='openpyxl')
    mg.columns = [str(c).strip() for c in mg.columns]
    mg = mg.dropna(subset=['Machine_number', 'Group']).copy()
    mg['Machine_number'] = mg['Machine_number'].apply(normalize_machine_name)
    mg['Group'] = pd.to_numeric(mg['Group'], errors='coerce').astype(int)
    g2m, m2g = defaultdict(list), {}
    for _, r in mg.iterrows():
        g2m[int(r['Group'])].append(r['Machine_number'])
        m2g[r['Machine_number']] = int(r['Group'])
    return dict(g2m), m2g


def load_sdst(path):
    sdst = pd.read_excel(path, sheet_name=0, engine='openpyxl')
    sdst.columns = [str(c).strip() for c in sdst.columns]
    setup = {}
    for _, r in sdst.dropna(subset=['diam_from', 'to_diam', 'setup_time']).iterrows():
        setup[(float(r['diam_from']), float(r['to_diam']))] = float(r['setup_time'])
    return setup


def build_problem_data():
    ops    = load_operations(SHIPMENT_FILE)
    orders = load_orders(SHIPMENT_FILE)
    g2m, m2g = load_machine_groups(MACHINE_GROUP_FILE)
    setup_dict = load_sdst(SDST_FILE)

    part_diam = ops.groupby('part_no')['diameter'].agg(lambda x: mode_or_first(x, 0)).to_dict()
    eligible  = defaultdict(lambda: defaultdict(set))
    for _, r in ops.iterrows():
        eligible[r['part_no']][int(r['operation'])].add(int(r['machine_group']))

    unit_time = {}
    for key, g in ops.dropna(subset=['unit_min', 'machine_group']).groupby(['part_no', 'operation', 'machine_group']):
        unit_time[(str(key[0]), int(key[1]), int(key[2]))] = float(g['unit_min'].median())
    fb_part_op = ops.dropna(subset=['unit_min']).groupby(['part_no', 'operation'])['unit_min'].median().to_dict()
    fb_op      = ops.dropna(subset=['unit_min']).groupby('operation')['unit_min'].median().to_dict()
    global_unit = float(ops['unit_min'].dropna().median())

    all_groups = sorted(g2m.keys())
    jobs = []
    for _, r in orders.iterrows():
        pno  = str(r['part_no'])
        diam = float(part_diam.get(pno, ops['diameter'].dropna().median()))
        eg10 = sorted(eligible[pno].get(10, set(all_groups)))
        eg20 = sorted(eligible[pno].get(20, set(eg10 if eg10 else all_groups)))
        if not eg10: eg10 = all_groups
        if not eg20: eg20 = eg10
        jobs.append(Job(job_id=r['job_id'], part_no=pno, due_date=r['due_date'],
                        quantity=int(r['quantity']), diameter=diam,
                        eligible_groups_op10=eg10, eligible_groups_op20=eg20))

    baseline = float(ops.get('overtime_text', pd.Series(dtype=object)).apply(parse_overtime_text).sum())
    ps = datetime.combine(min(j.due_date.date() for j in jobs), dtime(PLANNING_START_HOUR, 0))

    return {'ops': ops, 'baseline_overtime_min': baseline, 'jobs': jobs,
            'group_to_machines': g2m, 'machine_to_group': m2g,
            'setup_dict': setup_dict, 'unit_time': unit_time,
            'fallback_part_op': fb_part_op, 'fallback_op': fb_op,
            'global_unit': global_unit, 'planning_start': ps}


def get_unit_time(data, part_no, operation, group):
    k = (str(part_no), int(operation), int(group))
    if k in data['unit_time']: return data['unit_time'][k]
    k2 = (str(part_no), int(operation))
    if k2 in data['fallback_part_op']: return float(data['fallback_part_op'][k2])
    if int(operation) in data['fallback_op']: return float(data['fallback_op'][int(operation)])
    return data['global_unit']


def get_setup_time(data, from_d, to_d):
    if from_d is None: return INITIAL_SETUP_MIN
    d1, d2 = float(from_d), float(to_d)
    if abs(d1 - d2) < 1e-9: return SAME_DIAM_SETUP_MIN
    return data['setup_dict'].get((d1, d2), DEFAULT_DIFF_DIAM_SETUP_MIN)


def common_groups(job):
    c = sorted(set(job.eligible_groups_op10) & set(job.eligible_groups_op20))
    return c if c else sorted(job.eligible_groups_op10)


# ============================================================
# 4. GUROBI MODEL
# ============================================================

def build_ot_windows(ps, latest_due):
    end = latest_due + timedelta(minutes=TARDINESS_LIMIT_MIN) + timedelta(days=1)
    windows, day = [], ps.date()
    while datetime.combine(day, dtime(0, 0)) <= end:
        l = dt_to_min(datetime.combine(day, OVERTIME_START), ps)
        u = dt_to_min(datetime.combine(day, OVERTIME_END),   ps)
        if u > 0: windows.append((max(0.0, l), u, day.isoformat()))
        day += timedelta(days=1)
    return windows


def solve(data):
    jobs = data['jobs']
    ps   = data['planning_start']
    latest_due = max(j.due_date for j in jobs)
    ot_windows = build_ot_windows(ps, latest_due)
    machines   = sorted(data['machine_to_group'].keys())
    split_ids  = list(range(1, (MAX_SPLITS if ALLOW_JOB_SPLITTING else 1) + 1))

    tasks = [(j.job_id, r, op) for j in jobs for r in split_ids for op in [10, 20]]
    job_by_id    = {j.job_id: j for j in jobs}
    allowed_grps = {j.job_id: common_groups(j) for j in jobs}
    elig_mach    = {}
    for j in jobs:
        for r in split_ids:
            for op in [10, 20]:
                ms = []
                for g in allowed_grps[j.job_id]:
                    ms.extend(data['group_to_machines'].get(g, []))
                elig_mach[(j.job_id, r, op)] = sorted(set(ms))

    # Compute a tight horizon for Big-M validation
    horizon = dt_to_min(latest_due + timedelta(minutes=TARDINESS_LIMIT_MIN) + timedelta(days=1), ps)
    print(f'\nSchedule horizon : {horizon:.0f} min  ({horizon/60:.1f} h)')
    print(f'Big-M used       : {BIG_M}  (was 10**7 = {10**7})')
    print(f'Tightening ratio : {10**7 / BIG_M:.0f}x smaller')

    # Use the configured BIG_M (already set to 50,000 above)
    M = BIG_M

    model = Model('Exact_Medium')

    # --- NEW: Gurobi log file ---
    model.Params.LogFile    = str(GUROBI_LOG_FILE)
    # ----------------------------

    model.Params.MIPGap     = MIP_GAP
    model.Params.OutputFlag = OUTPUT_FLAG
    if TIME_LIMIT_SEC:
        model.Params.TimeLimit = TIME_LIMIT_SEC

    # Variables
    active   = model.addVars([(j.job_id, r) for j in jobs for r in split_ids], vtype=GRB.BINARY,  name='active')
    qty      = model.addVars([(j.job_id, r) for j in jobs for r in split_ids], vtype=GRB.INTEGER, lb=0, name='qty')
    z_group  = model.addVars([(j.job_id, r, g) for j in jobs for r in split_ids for g in allowed_grps[j.job_id]], vtype=GRB.BINARY,  name='z_group')
    qty_grp  = model.addVars([(j.job_id, r, g) for j in jobs for r in split_ids for g in allowed_grps[j.job_id]], vtype=GRB.INTEGER, lb=0, name='qty_grp')
    x        = model.addVars([(t[0], t[1], t[2], m) for t in tasks for m in elig_mach[t]], vtype=GRB.BINARY, name='x')
    S        = model.addVars(tasks, lb=0.0, name='S')
    C        = model.addVars(tasks, lb=0.0, name='C')
    proc_t   = model.addVars(tasks, lb=0.0, name='proc_t')

    arc_keys = [(t[0], t[1], t[2], u[0], u[1], u[2], m)
                for t in tasks for u in tasks if t != u
                for m in set(elig_mach[t]) & set(elig_mach[u])]
    arc   = model.addVars(arc_keys, vtype=GRB.BINARY, name='arc')
    first = model.addVars([(t[0], t[1], t[2], m) for t in tasks for m in elig_mach[t]], vtype=GRB.BINARY, name='first')
    last  = model.addVars([(t[0], t[1], t[2], m) for t in tasks for m in elig_mach[t]], vtype=GRB.BINARY, name='last')

    job_finish = model.addVars([j.job_id for j in jobs], lb=0.0, name='job_finish')
    makespan   = model.addVar(lb=0.0, name='makespan')

    ot_vars    = {(t, wi): model.addVar(lb=0.0, name=f'ot_{t[0]}_{t[1]}_{t[2]}_{wi}')
                  for t in tasks for wi in range(len(ot_windows))}
    start_side = {}
    if DISALLOW_START_IN_OVERTIME:
        start_side = {(t, wi): model.addVar(vtype=GRB.BINARY, name=f'ss_{t[0]}_{t[1]}_{t[2]}_{wi}')
                      for t in tasks for wi in range(len(ot_windows))}

    # Constraints: splits
    for j in jobs:
        jid, Q = j.job_id, int(j.quantity)
        min_q  = min(MIN_SPLIT_QTY, Q)
        model.addConstr(active[jid, 1] == 1)
        model.addConstr(quicksum(qty[jid, r] for r in split_ids) == Q)
        for r in split_ids:
            if r > 1:
                model.addConstr(active[jid, r] <= active[jid, r - 1])
                if Q < 2 * MIN_SPLIT_QTY:
                    model.addConstr(active[jid, r] == 0)
            model.addConstr(qty[jid, r] <= Q * active[jid, r])
            model.addConstr(qty[jid, r] >= min_q * active[jid, r])
            model.addConstr(quicksum(z_group[jid, r, g] for g in allowed_grps[jid]) == active[jid, r])
            model.addConstr(quicksum(qty_grp[jid, r, g] for g in allowed_grps[jid]) == qty[jid, r])
            for g in allowed_grps[jid]:
                model.addConstr(qty_grp[jid, r, g] <= Q * z_group[jid, r, g])
                model.addConstr(qty_grp[jid, r, g] <= qty[jid, r])
                model.addConstr(qty_grp[jid, r, g] >= qty[jid, r] - Q * (1 - z_group[jid, r, g]))

    # Constraints: assignment
    for t in tasks:
        jid, r, op = t
        model.addConstr(quicksum(x[jid, r, op, m] for m in elig_mach[t]) == active[jid, r])
        for m in elig_mach[t]:
            gm = data['machine_to_group'][m]
            if gm in allowed_grps[jid]:
                model.addConstr(x[jid, r, op, m] <= z_group[jid, r, gm])
            else:
                model.addConstr(x[jid, r, op, m] == 0)

    # Constraints: processing time, completion, precedence
    for t in tasks:
        jid, r, op = t
        job = job_by_id[jid]
        p_expr = quicksum(get_unit_time(data, job.part_no, op, g) * qty_grp[jid, r, g]
                          for g in allowed_grps[jid])
        model.addConstr(proc_t[t] == p_expr)
        model.addConstr(C[t] == S[t] + proc_t[t])
        model.addConstr(S[t] <= M * active[jid, r])
        model.addConstr(C[t] <= M * active[jid, r])

    for j in jobs:
        for r in split_ids:
            model.addConstr(S[j.job_id, r, 20] >= C[j.job_id, r, 10] - M * (1 - active[j.job_id, r]))

    for j in jobs:
        due_m = dt_to_min(j.due_date, ps)
        for r in split_ids:
            model.addConstr(job_finish[j.job_id] >= C[j.job_id, r, 20])
        model.addConstr(job_finish[j.job_id] <= due_m + TARDINESS_LIMIT_MIN)
        model.addConstr(makespan >= job_finish[j.job_id])

    # Constraints: machine sequencing
    for m in machines:
        pm = [t for t in tasks if m in elig_mach[t]]
        if not pm: continue
        model.addConstr(quicksum(first[t[0], t[1], t[2], m] for t in pm) <= 1)
        model.addConstr(quicksum(last[t[0], t[1], t[2], m]  for t in pm) <= 1)
        for t in pm:
            tid = (t[0], t[1], t[2], m)
            pred_a = [arc[u[0], u[1], u[2], t[0], t[1], t[2], m] for u in pm if u != t and (u[0], u[1], u[2], t[0], t[1], t[2], m) in arc]
            succ_a = [arc[t[0], t[1], t[2], u[0], u[1], u[2], m] for u in pm if u != t and (t[0], t[1], t[2], u[0], u[1], u[2], m) in arc]
            model.addConstr(quicksum(pred_a) + first[tid] == x[tid])
            model.addConstr(quicksum(succ_a) + last[tid]  == x[tid])
            model.addConstr(first[tid] <= x[tid])
            model.addConstr(last[tid]  <= x[tid])
            model.addConstr(S[t] >= INITIAL_SETUP_MIN - M * (1 - first[tid]))

    for key in arc_keys:
        i_jid, i_r, i_op, j_jid, j_r, j_op, m = key
        ti, tj = (i_jid, i_r, i_op), (j_jid, j_r, j_op)
        model.addConstr(arc[key] <= x[i_jid, i_r, i_op, m])
        model.addConstr(arc[key] <= x[j_jid, j_r, j_op, m])
        s_ij = get_setup_time(data, job_by_id[i_jid].diameter, job_by_id[j_jid].diameter)
        model.addConstr(S[tj] >= C[ti] + s_ij - M * (1 - arc[key]))

    # Constraints: overtime overlap
    for t in tasks:
        jid, r, op = t
        for wi, (l, u, _) in enumerate(ot_windows):
            ms = model.addVar(lb=-M, ub=M, name=f'ms_{jid}_{r}_{op}_{wi}')
            mf = model.addVar(lb=-M, ub=M, name=f'mf_{jid}_{r}_{op}_{wi}')
            df = model.addVar(lb=-M, ub=M, name=f'df_{jid}_{r}_{op}_{wi}')
            model.addGenConstrMax(ms, [S[t]], constant=float(l))
            model.addGenConstrMin(mf, [C[t]], constant=float(u))
            model.addConstr(df == mf - ms)
            model.addGenConstrMax(ot_vars[(t, wi)], [df], constant=0.0)
            model.addConstr(ot_vars[(t, wi)] <= (u - l) * active[jid, r])
            if DISALLOW_START_IN_OVERTIME:
                b = start_side[(t, wi)]
                model.addConstr(S[t] <= float(l) + M * b + M * (1 - active[jid, r]))
                model.addConstr(S[t] >= float(u) - M * (1 - b) - M * (1 - active[jid, r]))

    total_ot    = quicksum(ot_vars[(t, wi)] for t in tasks for wi in range(len(ot_windows)))
    total_setup = (quicksum(INITIAL_SETUP_MIN * first[k] for k in first.keys()) +
                   quicksum(get_setup_time(data, job_by_id[k[0]].diameter, job_by_id[k[3]].diameter) * arc[k]
                            for k in arc_keys))

    model.setObjective(W_OVERTIME * total_ot + W_MAKESPAN * makespan + W_SETUP * total_setup, GRB.MINIMIZE)
    model.update()

    print(f'\nModel size:')
    print(f'  Variables   : {model.NumVars}')
    print(f'  Binary vars : {sum(1 for v in model.getVars() if v.VType == GRB.BINARY)}')
    print(f'  Integer vars: {sum(1 for v in model.getVars() if v.VType == GRB.INTEGER)}')
    print(f'  Continuous  : {sum(1 for v in model.getVars() if v.VType == GRB.CONTINUOUS)}')
    print(f'  Constraints : {model.NumConstrs}')
    print(f'  GenConstrs  : {model.NumGenConstrs}')
    print(f'\nGurobi log   : {GUROBI_LOG_FILE}')
    print(f'Time limit   : {TIME_LIMIT_SEC} sec ({TIME_LIMIT_SEC/3600:.0f} hours)')

    model.optimize()

    return model, {
        'active': active, 'qty': qty, 'z_group': z_group, 'qty_grp': qty_grp,
        'x': x, 'S': S, 'C': C, 'proc_t': proc_t, 'arc': arc,
        'first': first, 'last': last, 'job_finish': job_finish, 'makespan': makespan,
        'ot_vars': ot_vars, 'ot_windows': ot_windows, 'tasks': tasks,
        'machines': machines, 'split_ids': split_ids,
        'allowed_grps': allowed_grps, 'elig_mach': elig_mach, 'job_by_id': job_by_id,
    }


# ============================================================
# 5. EXTRACT & EXPORT
# ============================================================

def val(v):
    try: return float(v.X)
    except: return float(v)


def extract_schedule(data, sol):
    jobs, ps = data['jobs'], data['planning_start']
    rows = []
    for t in sol['tasks']:
        jid, r, op = t
        if val(sol['active'][jid, r]) <= 0.5: continue
        job = sol['job_by_id'][jid]
        m_sel = next((m for m in sol['elig_mach'][t] if (jid, r, op, m) in sol['x'] and val(sol['x'][jid, r, op, m]) > 0.5), None)
        if m_sel is None: continue
        is_first = val(sol['first'][jid, r, op, m_sel]) > 0.5
        setup = INITIAL_SETUP_MIN if is_first else next(
            (get_setup_time(data, sol['job_by_id'][k[0]].diameter, job.diameter)
             for k, v in sol['arc'].items() if k[3] == jid and k[4] == r and k[5] == op and k[6] == m_sel and val(v) > 0.5), 0.0)
        g_sel = next((g for g in sol['allowed_grps'][jid] if val(sol['z_group'][jid, r, g]) > 0.5), None)
        s_min = val(sol['S'][t]); f_min = val(sol['C'][t])
        q_sel = round(val(sol['qty'][jid, r]))
        rows.append({
            'job_id': jid, 'part_no': job.part_no, 'split_id': r,
            'quantity': q_sel, 'diameter': job.diameter, 'group': g_sel,
            'operation': op, 'machine': m_sel,
            'start_min': s_min, 'finish_min': f_min,
            'processing_min': val(sol['proc_t'][t]), 'setup_min': setup,
            'overtime_min': overtime_overlap(min_to_dt(s_min, ps), min_to_dt(f_min, ps)),
            'start': min_to_dt(s_min, ps), 'finish': min_to_dt(f_min, ps),
            'due_date': job.due_date,
        })
    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values(['machine', 'start_min']).reset_index(drop=True)
    return df


def export_results(data, sol, schedule_df, cpu_seconds, peak_mb):
    with pd.ExcelWriter(OUTPUT_SCHEDULE_FILE, engine='openpyxl') as writer:
        schedule_df.to_excel(writer, sheet_name='Schedule', index=False)
        metrics = pd.DataFrame([{
            'gurobi_status':      sol[0].Status,
            'mip_gap_pct':        round(sol[0].MIPGap * 100, 4) if sol[0].Status in [2, 9, 11] else None,
            'obj_value':          round(sol[0].ObjVal, 4) if sol[0].SolCount > 0 else None,
            'best_bound':         round(sol[0].ObjBound, 4) if sol[0].SolCount > 0 else None,
            'total_overtime_min': round(schedule_df['overtime_min'].sum(), 2) if not schedule_df.empty else None,
            'solution_count':     sol[0].SolCount,
            'cpu_time_seconds':   round(cpu_seconds, 2),
            'cpu_time_hours':     round(cpu_seconds / 3600, 4),
            'peak_memory_mb':     round(peak_mb, 2),
            'peak_memory_gb':     round(peak_mb / 1024, 3),
            'big_m_used':         BIG_M,
            'tardiness_limit_days': TARDINESS_LIMIT_DAYS,
            'time_limit_sec':     TIME_LIMIT_SEC,
            'gurobi_log':         str(GUROBI_LOG_FILE),
        }])
        metrics.to_excel(writer, sheet_name='Metrics', index=False)
    print(f'\nResults saved  : {OUTPUT_SCHEDULE_FILE}')
    print(f'Gurobi log     : {GUROBI_LOG_FILE}')


# ============================================================
# 6. MAIN
# ============================================================

if __name__ == '__main__':
    tracemalloc.start()
    process = psutil.Process(os.getpid())
    start_time = time.time()

    print('\nLoading data...')
    data = build_problem_data()
    print(f"Jobs          : {len(data['jobs'])}")
    print(f"Machines      : {sorted(data['machine_to_group'].keys())}")
    print(f"Planning start: {data['planning_start']}")

    print('\nSolving...')
    model, sol_vars = solve(data)

    cpu_seconds = time.time() - start_time
    _, peak_traced = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    peak_mb_traced = peak_traced / 1024 / 1024
    peak_mb_psutil = process.memory_info().rss / 1024 / 1024
    peak_mb = max(peak_mb_traced, peak_mb_psutil)

    print(f'\n{"="*55}')
    print(f'CPU Time       : {cpu_seconds:.2f} sec  ({cpu_seconds/3600:.3f} hours)')
    print(f'Peak Memory    : {peak_mb:.2f} MB  ({peak_mb/1024:.3f} GB)')
    print(f'Gurobi Status  : {model.Status}')
    print(f'Solution Count : {model.SolCount}')
    if model.SolCount > 0:
        print(f'Objective      : {model.ObjVal:.4f}')
        print(f'Best Bound     : {model.ObjBound:.4f}')
        print(f'MIP Gap        : {model.MIPGap*100:.2f}%')
    print(f'Big-M used     : {BIG_M}  (revised from 10**7)')
    print(f'Tardiness limit: {TARDINESS_LIMIT_DAYS} days = {TARDINESS_LIMIT_MIN:.0f} min')
    print(f'Time limit     : {TIME_LIMIT_SEC} sec = {TIME_LIMIT_SEC/3600:.0f} hours')
    print(f'Gurobi log     : {GUROBI_LOG_FILE}')
    print(f'{"="*55}')

    sol = (model, sol_vars)

    if model.SolCount > 0:
        print('\nExtracting schedule...')
        schedule_df = extract_schedule(data, sol_vars)
        print(f'Scheduled tasks: {len(schedule_df)}')
        export_results(data, sol, schedule_df, cpu_seconds, peak_mb)
    else:
        print('\nNo feasible solution found.')
        metrics_only = pd.DataFrame([{
            'gurobi_status':    model.Status,
            'solution_count':   model.SolCount,
            'best_bound':       model.ObjBound if hasattr(model, 'ObjBound') else None,
            'cpu_time_seconds': round(cpu_seconds, 2),
            'cpu_time_hours':   round(cpu_seconds / 3600, 4),
            'peak_memory_mb':   round(peak_mb, 2),
            'big_m_used':       BIG_M,
            'tardiness_limit_days': TARDINESS_LIMIT_DAYS,
            'time_limit_sec':   TIME_LIMIT_SEC,
        }])
        with pd.ExcelWriter(OUTPUT_SCHEDULE_FILE, engine='openpyxl') as writer:
            metrics_only.to_excel(writer, sheet_name='Metrics', index=False)
        print(f'Metrics saved  : {OUTPUT_SCHEDULE_FILE}')
        print(f'Gurobi log     : {GUROBI_LOG_FILE}')

Using files:
SHIPMENT_FILE      = C:\Users\emrec\Documents\parallel_scheduling\exactMEDIUM\sevkiyat_medium_selected8.xlsx
SDST_FILE          = C:\Users\emrec\Documents\parallel_scheduling\exactMEDIUM\SDST.xlsx
MACHINE_GROUP_FILE = C:\Users\emrec\Documents\parallel_scheduling\exactMEDIUM\machine_group_data_medium.xlsx
GUROBI_LOG_FILE    = C:\Users\emrec\Documents\parallel_scheduling\exactMEDIUM\gurobi_exact_medium.log

Loading data...
Jobs          : 8
Machines      : ['T.3.10', 'T.3.6', 'T.3.7', 'T.3.9']
Planning start: 2026-04-29 07:00:00

Solving...

Schedule horizon : 14280 min  (238.0 h)
Big-M used       : 50000  (was 10**7 = 10000000)
Tightening ratio : 200x smaller
Set parameter Username
Set parameter LicenseID to value 2797590
Academic license - for non-commercial use only - expires 2027-03-24
Set parameter LogFile to value "C:\Users\emrec\Documents\parallel_scheduling\exactMEDIUM\gurobi_exact_medium.log"
Set parameter MIPGap to value 0.05
Set parameter OutputFlag to value 1
Set

 44723 22428    9.17520  110   93    9.52284    5.13935  46.0%  41.6  115s
 44731 22434    8.93962  120  104    9.52284    5.13968  46.0%  41.6  120s
 44738 22438    5.24467   85   90    9.52284    5.15363  45.9%  41.6  125s
 44742 22441    5.39517   56   97    9.52284    5.15363  45.9%  41.6  130s
 44745 22443    5.29467   82   85    9.52284    5.16493  45.8%  41.6  135s
 44748 22445    6.27548  101   54    9.52284    5.16583  45.8%  41.6  141s
 44751 22447    6.70901  114   79    9.52284    5.16587  45.8%  41.6  145s
 44754 22449    5.77401  146   64    9.52284    5.16587  45.8%  41.6  151s
 44757 22451    6.53279  132   71    9.52284    5.16613  45.8%  41.6  155s
 44763 22455    7.26238  106  104    9.52284    5.16716  45.7%  41.6  160s
 44769 22459    6.55384   83   91    9.52284    5.16893  45.7%  41.5  165s
 44776 22464    5.79773  141   48    9.52284    5.17053  45.7%  41.5  170s
 44782 22468    8.68873   97   61    9.52284    5.17053  45.7%  41.5  175s
 44789 22475    5.17054  


Results saved  : C:\Users\emrec\Documents\parallel_scheduling\exactMEDIUM\optimized_exact_medium_schedule.xlsx
Gurobi log     : C:\Users\emrec\Documents\parallel_scheduling\exactMEDIUM\gurobi_exact_medium.log
